# Assignment 4 - Autoencoder anomaly detection (MNIST)
**Offline version** - runs without internet.
Needs `mnist.npz` in the same folder (see `00_download_once.ipynb`).

In [ ]:
# ============================================================
# ASSIGNMENT 4: AUTOENCODER FOR ANOMALY DETECTION
# MNIST: Digit 1 = Normal, Other Digits = Anomalies
# ============================================================

import numpy as np

# ---- OFFLINE MNIST LOADER (no internet needed) --------------------------------
# Reads mnist.npz from the notebook folder (or from ~/.keras/datasets).
# Get the file once from: https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz
import os
def load_mnist(path="mnist.npz"):
    for p in (path, os.path.expanduser("~/.keras/datasets/mnist.npz")):
        if os.path.exists(p):
            with np.load(p, allow_pickle=True) as f:
                return (f["x_train"], f["y_train"]), (f["x_test"], f["y_test"])
    raise FileNotFoundError("mnist.npz not found. Put it in the same folder as this notebook "
        "(download link: https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz)")
# -------------------------------------------------------------------------------

import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)


# ============================================================
# B. LOAD AND PREPROCESS THE DATASET
# ============================================================

(x_train, y_train), (x_test, y_test) = load_mnist()

# Flatten images and normalize pixel values
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0

NORMAL = 1

# Train only on normal digit 1
x_normal = x_train[y_train == NORMAL]

# Select normal test images and 150 anomalous images
rng = np.random.default_rng(42)

norm_idx = np.where(y_test == NORMAL)[0]

anom_idx = rng.choice(
    np.where(y_test != NORMAL)[0],
    size=150,
    replace=False
)

idx = np.concatenate([norm_idx, anom_idx])
rng.shuffle(idx)

x_eval = x_test[idx]

# 0 = normal, 1 = anomaly
y_eval = (y_test[idx] != NORMAL).astype(int)

print("Normal training samples:", x_normal.shape[0])
print("Total test samples:", len(idx))
print("Normal test samples:", np.sum(y_eval == 0))
print("Anomaly test samples:", np.sum(y_eval == 1))


# ============================================================
# C & D. BUILD THE ENCODER AND DECODER
# ============================================================

# Encoder
inp = keras.Input(shape=(784,))

e = layers.Dense(128, activation="relu")(inp)
e = layers.Dense(64, activation="relu")(e)

# Latent representation
code = layers.Dense(
    16,
    activation="relu",
    name="latent"
)(e)

# Decoder
d = layers.Dense(64, activation="relu")(code)
d = layers.Dense(128, activation="relu")(d)

out = layers.Dense(784, activation="sigmoid")(d)

# Complete autoencoder
autoencoder = keras.Model(
    inp,
    out,
    name="MNIST_Autoencoder"
)

print("\nAutoencoder Model Summary:")
autoencoder.summary()


# ============================================================
# E. COMPILE THE MODEL
# ============================================================

autoencoder.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)


# ============================================================
# F. TRAIN THE AUTOENCODER
# ============================================================

history = autoencoder.fit(
    x_normal,
    x_normal,
    epochs=30,
    batch_size=128,
    validation_split=0.1,
    shuffle=True,
    verbose=2
)


# ============================================================
# RECONSTRUCTION ERROR AND ANOMALY DETECTION
# ============================================================

def rec_error(x):
    reconstructed = autoencoder.predict(x, verbose=0)
    return np.mean((x - reconstructed) ** 2, axis=1)


# Reconstruction errors for training images
train_err = rec_error(x_normal)

# Set threshold using normal training errors
threshold = train_err.mean() + 2 * train_err.std()

# Reconstruction errors for test images
err = rec_error(x_eval)

# Predict anomalies
y_pred = (err > threshold).astype(int)


# ============================================================
# EVALUATION METRICS
# ============================================================

print("\nThreshold (mean + 2*std):", round(threshold, 5))

print(
    "Mean reconstruction error - normal:",
    round(err[y_eval == 0].mean(), 5)
)

print(
    "Mean reconstruction error - anomaly:",
    round(err[y_eval == 1].mean(), 5)
)

print("\nROC-AUC Score:", round(roc_auc_score(y_eval, err), 4))

print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_eval, y_pred, labels=[0, 1]))

print("\nClassification Report:")
print(
    classification_report(
        y_eval,
        y_pred,
        labels=[0, 1],
        target_names=["normal", "anomaly"],
        digits=4,
        zero_division=0
    )
)


# ============================================================
# GRAPH 1: TRAINING LOSS AND RECONSTRUCTION ERROR
# ============================================================

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# Training and validation loss
ax[0].plot(
    history.history["loss"],
    label="train_loss"
)

ax[0].plot(
    history.history["val_loss"],
    label="val_loss"
)

ax[0].set_title("Autoencoder training loss (MSE)")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].legend()
ax[0].grid(alpha=0.3)

# Reconstruction error histogram
ax[1].hist(
    err[y_eval == 0],
    bins=50,
    alpha=0.7,
    label="normal"
)

ax[1].hist(
    err[y_eval == 1],
    bins=50,
    alpha=0.7,
    label="anomaly"
)

ax[1].axvline(
    threshold,
    color="black",
    linestyle="--",
    label="threshold"
)

ax[1].set_title("Reconstruction error")
ax[1].set_xlabel("MSE")
ax[1].set_ylabel("Count")
ax[1].legend()
ax[1].grid(alpha=0.2)

plt.tight_layout()

# Save and display graph once
plt.savefig("a4_error.png", dpi=130, bbox_inches="tight")
plt.show()
plt.close(fig)


# ============================================================
# GRAPH 2: ORIGINAL AND RECONSTRUCTED IMAGES
# ============================================================

# Select five normal and five anomalous test images
normal_samples = np.where(y_eval == 0)[0][:5]
anomaly_samples = np.where(y_eval == 1)[0][:5]

sel = list(normal_samples) + list(anomaly_samples)

# Reconstruct selected images
rec = autoencoder.predict(x_eval[sel], verbose=0)

fig, ax = plt.subplots(2, 10, figsize=(12, 2.8))

for i, k in enumerate(sel):

    # Original input
    ax[0, i].imshow(
        x_eval[k].reshape(28, 28),
        cmap="gray",
        vmin=0,
        vmax=1
    )
    ax[0, i].axis("off")

    # Reconstructed output
    ax[1, i].imshow(
        rec[i].reshape(28, 28),
        cmap="gray",
        vmin=0,
        vmax=1
    )
    ax[1, i].axis("off")

    # Label the images
    ax[0, i].set_title(
        "normal" if y_eval[k] == 0 else "anomaly",
        fontsize=8
    )

# Row labels
ax[0, 0].text(
    -0.3, 0.5, "input",
    transform=ax[0, 0].transAxes,
    rotation=90,
    va="center",
    ha="right",
    fontsize=8
)

ax[1, 0].text(
    -0.3, 0.5, "output",
    transform=ax[1, 0].transAxes,
    rotation=90,
    va="center",
    ha="right",
    fontsize=8
)

plt.tight_layout()

# Save and display graph once
plt.savefig("a4_recon.png", dpi=130, bbox_inches="tight")
plt.show()
plt.close(fig)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\nAssignment completed successfully!")
print("Graph 1 saved as: a4_error.png")
print("Graph 2 saved as: a4_recon.png")